# Setup

In [15]:
from platform import python_version
print(python_version())

3.10.20


In [16]:
import tensorboard
print("TensorBoard works!")

TensorBoard works!


In [17]:
import timm
print("timm works!")

timm works!


In [18]:
import os
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import random_split, DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms, models
from torch.utils.tensorboard import SummaryWriter
import timm

# Reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cpu


# Data Loading and Splitting

In [19]:
import os
# Set correct path
data_root = "C:/Users/USER/Downloads/The IQ-OTHNCCD lung cancer dataset"
# Check if path exists
print("Path exists:", os.path.exists(data_root))

# Defining transforms
common_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Loading dataset
dataset = ImageFolder(root=data_root, transform=common_transform)

class_names = dataset.classes
print(f"Classes: {class_names}, Total images: {len(dataset)}")

# Splitting sizes
train_size = int(0.8 * len(dataset))
val_size = int(0.1 * len(dataset))
test_size = len(dataset) - train_size - val_size

# Splitting
train_dataset, val_dataset, test_dataset = random_split(
    dataset,
    [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(seed)
)

print(f"Train: {len(train_dataset)}, Val: {len(val_dataset)}, Test: {len(test_dataset)}")


Path exists: True
Classes: ['Bengin cases', 'Malignant cases', 'Normal cases'], Total images: 1097
Train: 877, Val: 109, Test: 111


# Preprocessing and Data Augmentation

In [20]:
# Training augmentations
train_transform = transforms.Compose([
    transforms.Resize((256, 256)),
    transforms.RandomRotation(degrees=15),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Validation/Test transforms (no random augmentations)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Transforms to the split subsets
train_dataset.dataset.transform = train_transform
val_dataset.dataset.transform = val_transform
test_dataset.dataset.transform = val_transform

# DataLoaders
batch_size = 16  # example batch size
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True,  pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=batch_size, shuffle=False, pin_memory=True)
test_loader  = DataLoader(test_dataset,  batch_size=batch_size, shuffle=False, pin_memory=True)


In [21]:
import os
import random
import numpy as np

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim

from torchvision import models
from torch.utils.data import DataLoader
from torchvision.datasets import ImageFolder
from torchvision import transforms

from torch.utils.tensorboard import SummaryWriter
import timm

# Reproducibility

seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)

# Device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)



# Image transforms (basic safe version)

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor()
])


# 1. MobileNetV2

mobilenetv2 = models.mobilenet_v2(
    weights=models.MobileNet_V2_Weights.DEFAULT
)

mobilenetv2.classifier[1] = nn.Linear(
    mobilenetv2.classifier[1].in_features,
    3
)

mobilenetv2 = mobilenetv2.to(device)


# 2. EfficientNet-B0

effnet_b0 = models.efficientnet_b0(
    weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1
)

effnet_b0.classifier[1] = nn.Linear(
    effnet_b0.classifier[1].in_features,
    3
)

effnet_b0 = effnet_b0.to(device)



# 3. DeiT-Tiny

deit_tiny = timm.create_model(
    'deit_tiny_distilled_patch16_224',
    pretrained=True,
    num_classes=3
).to(device)



# 4. DeiT-Small

deit_small = timm.create_model(
    'deit_small_patch16_224',
    pretrained=True,
    num_classes=3
).to(device)



# 5. Swin-Tiny

swin_tiny = timm.create_model(
    'swin_tiny_patch4_window7_224',
    pretrained=True,
    num_classes=3
).to(device)

# Summary

models_list = [
    mobilenetv2,
    effnet_b0,
    deit_tiny,
    deit_small,
    swin_tiny
]

print("\nAll models loaded successfully!")
for i, m in enumerate(models_list):
    print(f"Model {i+1}: {m.__class__.__name__}")

Using device: cpu

All models loaded successfully!
Model 1: MobileNetV2
Model 2: EfficientNet
Model 3: VisionTransformerDistilled
Model 4: VisionTransformer
Model 5: SwinTransformer


# Training Loop

In [26]:
import torch.nn as nn

criterion = nn.CrossEntropyLoss()

In [27]:
models_dict = {
    "mobilenet": mobilenetv2,
    "effnet": effnet_b0,
    "deit_tiny": deit_tiny,
    "deit_small": deit_small,
    "swin": swin_tiny
}

In [28]:
def train_one_epoch(model, loader, optimizer):
    model.train()
    running_loss, running_correct = 0.0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        preds = outputs.argmax(dim=1)
        running_correct += (preds == labels).sum().item()

    epoch_loss = running_loss / len(loader.dataset)
    epoch_acc = 100 * running_correct / len(loader.dataset)

    return epoch_loss, epoch_acc

In [29]:
def evaluate(model, loader):
    model.eval()
    val_loss, val_correct = 0.0, 0

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item() * images.size(0)
            val_correct += (outputs.argmax(dim=1) == labels).sum().item()

    loss = val_loss / len(loader.dataset)
    acc = 100 * val_correct / len(loader.dataset)

    return loss, acc

In [30]:
num_epochs = 5

for name, model in models_dict.items():

    print(f"Training: {name}")
    print(f"====================")

    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)

    best_val_acc = 0

    for epoch in range(1, num_epochs + 1):

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        val_loss, val_acc = evaluate(model, val_loader)

        print(f"{name} Epoch {epoch}: "
              f"Train Acc {train_acc:.2f}% | "
              f"Val Acc {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"{name}_best.pth")

print("\nTraining complete for all models!")

Training: mobilenet
mobilenet Epoch 1: Train Acc 82.67% | Val Acc 91.74%
mobilenet Epoch 2: Train Acc 88.48% | Val Acc 91.74%
mobilenet Epoch 3: Train Acc 93.96% | Val Acc 93.58%
mobilenet Epoch 4: Train Acc 98.97% | Val Acc 95.41%
mobilenet Epoch 5: Train Acc 99.77% | Val Acc 96.33%
Training: effnet
effnet Epoch 1: Train Acc 82.10% | Val Acc 91.74%
effnet Epoch 2: Train Acc 92.13% | Val Acc 97.25%
effnet Epoch 3: Train Acc 96.12% | Val Acc 100.00%
effnet Epoch 4: Train Acc 98.06% | Val Acc 99.08%
effnet Epoch 5: Train Acc 99.20% | Val Acc 99.08%
Training: deit_tiny
deit_tiny Epoch 1: Train Acc 80.96% | Val Acc 93.58%
deit_tiny Epoch 2: Train Acc 95.44% | Val Acc 95.41%
deit_tiny Epoch 3: Train Acc 98.63% | Val Acc 100.00%
deit_tiny Epoch 4: Train Acc 99.66% | Val Acc 99.08%
deit_tiny Epoch 5: Train Acc 99.54% | Val Acc 100.00%
Training: deit_small
deit_small Epoch 1: Train Acc 84.49% | Val Acc 90.83%
deit_small Epoch 2: Train Acc 94.75% | Val Acc 98.17%
deit_small Epoch 3: Train Acc 9

# Hyperparameter Tuning

In [ ]:
# def train_fn(config):
#     model = CustomCNN().to(device)
#     optimizer = optim.Adam(model.parameters(), lr=config["lr"])
#     # ... training loop with config["batch_size"], etc.
#     # tune.report(val_accuracy=val_acc)
# 
# search_space = {
#     "lr": tune.loguniform(1e-5, 1e-2),
#     "batch_size": tune.choice([8, 16, 32]),
#     "dropout": tune.uniform(0.2, 0.5)
# }
# tune.run(train_fn, config=search_space, num_samples=10, scheduler=ASHAScheduler())

# Fine-Tuning

In [31]:
import torch

finetune_lr = 1e-5

In [33]:
num_epochs = 3  # fine-tuning needs fewer epochs

for name, model in models_dict.items():

    print(f"Fine-tuning: {name}")

    model = model.to(device)

    # IMPORTANT: lower learning rate for fine-tuning
    optimizer = torch.optim.Adam(model.parameters(), lr=finetune_lr)

    best_val_acc = 0

    model.train()

    for epoch in range(1, num_epochs + 1):

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        val_loss, val_acc = evaluate(model, val_loader)

        print(f"{name} FT Epoch {epoch}: "
              f"Train Acc {train_acc:.2f}% | Val Acc {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"{name}_finetuned_best.pth")

print("\nFine-tuning complete!")

Fine-tuning: mobilenet
mobilenet FT Epoch 1: Train Acc 99.89% | Val Acc 97.25%
mobilenet FT Epoch 2: Train Acc 99.89% | Val Acc 96.33%
mobilenet FT Epoch 3: Train Acc 99.89% | Val Acc 96.33%
Fine-tuning: effnet
effnet FT Epoch 1: Train Acc 98.97% | Val Acc 99.08%
effnet FT Epoch 2: Train Acc 99.77% | Val Acc 98.17%
effnet FT Epoch 3: Train Acc 99.54% | Val Acc 100.00%
Fine-tuning: deit_tiny
deit_tiny FT Epoch 1: Train Acc 99.89% | Val Acc 98.17%
deit_tiny FT Epoch 2: Train Acc 99.89% | Val Acc 100.00%
deit_tiny FT Epoch 3: Train Acc 100.00% | Val Acc 100.00%
Fine-tuning: deit_small
deit_small FT Epoch 1: Train Acc 99.89% | Val Acc 100.00%
deit_small FT Epoch 2: Train Acc 99.77% | Val Acc 99.08%
deit_small FT Epoch 3: Train Acc 99.89% | Val Acc 99.08%
Fine-tuning: swin
swin FT Epoch 1: Train Acc 99.77% | Val Acc 99.08%
swin FT Epoch 2: Train Acc 99.54% | Val Acc 99.08%
swin FT Epoch 3: Train Acc 99.89% | Val Acc 99.08%

Fine-tuning complete!


# Ensembling

In [35]:
import torch
import torch.nn.functional as F

models_list = [
    mobilenetv2,
    effnet_b0,
    deit_tiny,
    deit_small,
    swin_tiny
]

for m in models_list:
    m.eval()

correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = 0

        for m in models_list:
            outputs += F.softmax(m(images), dim=1)

        # average
        avg_output = outputs / len(models_list)

        preds = avg_output.argmax(dim=1)

        correct += (preds == labels).sum().item()
        total += labels.size(0)

accuracy = 100 * correct / total
print(f"5-Model Ensemble Accuracy: {accuracy:.2f}%")

5-Model Ensemble Accuracy: 100.00%


# Model Comparison

In [37]:
import torch

results = {}

for name, model in models_dict.items():
    model.load_state_dict(torch.load(f"{name}_best.pth"))
    model.to(device)
    model.eval()

    val_loss, val_acc = evaluate(model, val_loader)
    results[name] = val_acc

    print(f"{name}: {val_acc:.2f}%")

best_model_name = max(results, key=results.get)
print("\nBEST MODEL:", best_model_name)

mobilenet: 96.33%
effnet: 100.00%
deit_tiny: 100.00%
deit_small: 100.00%
swin: 98.17%

BEST MODEL: effnet


# Knowledge Distillation

In [38]:
import torch
import torch.nn.functional as F
import torch.optim as optim

In [39]:
best_model = effnet_b0  
best_model.load_state_dict(torch.load("effnet_best.pth"))
best_model.to(device)
best_model.eval()

stu_model = mobilenetv2
stu_model.to(device)
stu_model.train()

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [40]:
optimizer = optim.Adam(stu_model.parameters(), lr=1e-4)

In [42]:
T = 5.0      
alpha = 0.5  

In [45]:
num_epochs = 3

for epoch in range(num_epochs):

    stu_model.train()
    best_model.eval()

    running_loss = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        with torch.no_grad():
            teacher_logits = best_model(images)

        
        student_logits = stu_model(images)


        loss_ce = F.cross_entropy(student_logits, labels)

        
        loss_kd = F.kl_div(
            F.log_softmax(student_logits / T, dim=1),
            F.softmax(teacher_logits / T, dim=1),
            reduction='batchmean'
        ) * (T * T)

       
        loss = alpha * loss_ce + (1 - alpha) * loss_kd

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f"Epoch {epoch+1}/{num_epochs} | Loss: {running_loss:.4f}")

Epoch 1/3 | Loss: 6.8637
Epoch 2/3 | Loss: 5.0368
Epoch 3/3 | Loss: 4.9645


In [46]:
torch.save(stu_model.state_dict(), "student_distilled_model.pth")
print("Student model saved!")

Student model saved!


# Pruning

In [48]:
model_to_prune = stu_model 
model_to_prune.to(device)

MobileNetV2(
  (features): Sequential(
    (0): Conv2dNormActivation(
      (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
      (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (2): ReLU6(inplace=True)
    )
    (1): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(32, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), groups=32, bias=False)
          (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
          (2): ReLU6(inplace=True)
        )
        (1): Conv2d(32, 16, kernel_size=(1, 1), stride=(1, 1), bias=False)
        (2): BatchNorm2d(16, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      )
    )
    (2): InvertedResidual(
      (conv): Sequential(
        (0): Conv2dNormActivation(
          (0): Conv2d(16, 96, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (1): BatchNorm2d(96, eps=

In [49]:
parameters_to_prune = []

for module in model_to_prune.modules():
    if isinstance(module, (nn.Conv2d, nn.Linear)):
        parameters_to_prune.append((module, 'weight'))

In [50]:
prune.global_unstructured(
    parameters_to_prune,
    pruning_method=prune.L1Unstructured,
    amount=0.2  # 20% weights removed
)

In [51]:
for module, _ in parameters_to_prune:
    prune.remove(module, 'weight')

In [52]:
torch.save(model_to_prune.state_dict(), "pruned_student_model.pth")
print("Pruned model saved!")

Pruned model saved!


# Fine Tuning 

In [53]:
finetune_lr = 1e-5
num_epochs = 3

In [ ]:
for name, model in models_dict.items():

    print(f"Fine-tuning: {name}")
    print(f"====================")

    model = model.to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=finetune_lr)

    best_val_acc = 0

    for epoch in range(1, num_epochs + 1):

        model.train()

        train_loss, train_acc = train_one_epoch(model, train_loader, optimizer)
        val_loss, val_acc = evaluate(model, val_loader)

        print(f"{name} FT Epoch {epoch}: "
              f"Train Acc {train_acc:.2f}% | "
              f"Val Acc {val_acc:.2f}%")

        # save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            torch.save(model.state_dict(), f"{name}_finetuned_best.pth")

print("\nFine-tuning complete!")

Fine-tuning: mobilenet
mobilenet FT Epoch 1: Train Acc 99.89% | Val Acc 98.17%
mobilenet FT Epoch 2: Train Acc 100.00% | Val Acc 98.17%
mobilenet FT Epoch 3: Train Acc 100.00% | Val Acc 98.17%
Fine-tuning: effnet
effnet FT Epoch 1: Train Acc 98.40% | Val Acc 100.00%
effnet FT Epoch 2: Train Acc 98.18% | Val Acc 99.08%
effnet FT Epoch 3: Train Acc 98.40% | Val Acc 99.08%
Fine-tuning: deit_tiny
